# FDS Challenge: My Notebook

## 0. Configuration

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display
from pathlib import Path

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


DEBUG = False

pd.set_option("display.max_columns", None)  # keep it! it's usefull to make pandas print also the null columns
pd.set_option("display.width", None)


# --- PATHS ---
TRAIN_FILE_PATH        = "../dataset/train.jsonl"
TEST_FILE_PATH         = "../dataset/test.jsonl"
TYPE_CHART_PATH        = "../dataset/type_chart_gen2.csv"
STATS_MAP_PATH         = "../temp/stats_map.json"
SAMPLE_SUBMISSION_PATH = "../dataset/sample_submission.csv"
SUBMISSION_PATH        = "submission.csv"

Path("../temp").mkdir(exist_ok=True)


# --- FEATURES ---
TEAM_SIZE = 6

STATUSES_WEIGHTS = {
    "fnt": 7,
    "slp": 3,
    "frz": 3,
    "tox": 2,
    "psn": 1,
    "par": 2,
    "brn": 2
}

BOOSTS_WEIGHTS = {
    "atk": 1,
    "def": 1,
    "spa": 1,
    "spd": 1,
    "spe": 1
}

MULTS_WEIGHTS = {
    0:    -3,
    0.25: -2,
    0.5:  -1,
    1:     0,
    2:     1,
    4:     2
}

EFFECT_WEIGHTS = {
    "substitute":    1.25,
    "destinybond":   0.9,
    "perish":        0.15,
    "safeguard":     0.6,
    "bide":          0.4,
    "typechange":    0.3,
    "beatup":        0.3,
    "magnitude":     0.3,
    "lockon":        0.2,
    "noeffect":      0.0,
    "pursuit":      -1.5,
    "encore":       -1.2,
    "clamp":        -1.0,
    "nightmare":    -1.0,
    "trapped":      -0.9,
    "confusion":    -0.8,
    "leechseed":    -0.8,
    "protect":      -0.3,
    "attract":      -0.5,
    "curse":        -0.8,
    "whirlpool":    -0.4,
    "wrap":         -0.4,
    "firespin":     -0.4,
    "bind":         -0.4,
    "disable":      -0.4,
    "spite":        -0.3,
    "foresight":    -0.2,
    "endure":       -0.5,
    "focusenergy":  -0.4,
    "mimic":        -0.6,
    "futuresight":  -0.7,
    "mindreader":   -0.5,
    "focusband":    -0.8,
    "quickclaw":    -0.6,
    "mysteryberry": -0.3
}

DISABLED_FEATURES = [
    "p1_hp_pct_lost_0_10",
    "p1_hp_pct_lost_10_20",
    "p1_hp_pct_lost_20_30",
    "p2_hp_pct_lost_0_10",
    "p2_hp_pct_lost_10_20",
    "p2_hp_pct_lost_20_30"
]

# --- TRAINING ---
"""
"logistic_regression":
A simple linear model that weights each feature and outputs a win probability.
It is fast and easy to interpret, but it only captures straightforward relationships.
Features are standardized first.

"random_forest":
Trains 150 decision trees on different random subsets of the data and averages their votes.
It handles non-linear patterns well and is robust to noise and overfitting.

"lightgbm":
A gradient boosting model that builds trees one after another, each correcting the errors of the previous ones.
It is usually the most accurate on tabular data.

"ensemble":
A soft-voting combination of the three models above, averaging their probabilities with weights.
It tends to be the most stable, but it is the slowest because it trains three models per fold.
"""
MODEL_TYPE = "ensemble"
RANDOM_STATE = 42
K_FOLDS = 5
N_REPEATS = 3

## 1. Loading Data

In [ ]:
# --- LOAD DATA ---
def load_data(file_path: str) -> list:
    """
    Loads the whole `file_path` and stores it as a list of battles
    """

    print(f"Loading data from '{file_path}'...")
    data = []
    try:
        with open(file_path, "r") as f:
            for line in f:
                data.append(json.loads(line))
        print(f"SUCCESS: loaded {len(data)} battles.")
        
    except FileNotFoundError:
        print(f"ERROR: could not find the file at '{file_path}'.")

    return data


# Loading data
train_data = load_data(TRAIN_FILE_PATH)
test_data  = load_data(TEST_FILE_PATH)

### Pokemon Stats & Moves Type

In [ ]:
# --- POKEMON TYPE MAP ---
def create_stats_map(train_data: list, test_data: list, file_path: str = STATS_MAP_PATH) -> dict:
    """
    Build the type map, save it as JSON, and return it
    """

    print(f"Creating pokemon stats '{file_path}'...")
    stats_map = {}
    for data in (train_data, test_data):
        for battle in data:

            for pokemon in battle["p1_team_details"]:
                if pokemon["name"] not in stats_map:
                    stats_map[pokemon["name"]] = {"base_stats": {}, "types": []}
                stats_map[pokemon["name"]]["base_stats"] = {
                    "base_hp": pokemon["base_hp"],
                    "base_atk": pokemon["base_atk"],
                    "base_def": pokemon["base_def"],
                    "base_spa": pokemon["base_spa"],
                    "base_spd": pokemon["base_spd"],
                    "base_spe": pokemon["base_spe"],
                }
                stats_map[pokemon["name"]]["types"] = pokemon["types"]

            if battle["p2_lead_details"]["name"] not in stats_map:
                stats_map[battle["p2_lead_details"]["name"]] = {"base_stats": {}, "types": []}
            lead = battle["p2_lead_details"]
            stats_map[lead["name"]]["base_stats"] = {
                "base_hp": lead["base_hp"],
                "base_atk": lead["base_atk"],
                "base_def": lead["base_def"],
                "base_spa": lead["base_spa"],
                "base_spd": lead["base_spd"],
                "base_spe": lead["base_spe"],
            }
            stats_map[lead["name"]]["types"] = lead["types"]

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(stats_map, f, indent=4)

    print(f"SUCCESS: found {len(stats_map)} pokemon.")
    return stats_map


# --- MOVES TYPE JSON ---
def normalize_type_chart(type_chart: pd.DataFrame) -> pd.DataFrame:
    """
    Return a copy with trimmed column names and lowercase, trimmed types; reject missing types.
    """

    # Clear columns headers
    type_chart.columns = type_chart.columns.astype(str).str.strip()

    # Clear columns values
    for col in ["attacking_type", "defending_type"]:
        # If a value is Nan break the run
        if type_chart[col].isna().any():
            nan_rows = type_chart[type_chart[col].isna()].index.tolist()
            raise ValueError(f"ERROR: found NaN value in column '{col}' at rows {nan_rows}.")

        # Remove white spaces and lower
        type_chart[col] = type_chart[col].astype(str).str.strip().str.lower()

    print(f"SUCCESS: cleaned {len(type_chart)} entries.")
    return type_chart

def load_type_chart(file_path: str) -> pd.DataFrame | None:
    """
    Load and normalize a type chart CSV
    """

    print(f"Loading data from '{file_path}'...")
    try:
        raw_df = pd.read_csv(file_path)
        return normalize_type_chart(raw_df)
    
    except FileNotFoundError:
        print(f"ERROR: could not find the file at '{file_path}'.")


# --- MULTPLIERS --- 
def get_multiplier(move_type: str, pkm_type1: str, pkm_type2: str) -> float:
    """
    Return the move's combined effectiveness against the pokemon's types
    """

    m1 = type_chart.get((move_type, pkm_type1), 1.0) if pkm_type1 else 1.0
    m2 = type_chart.get((move_type, pkm_type2), 1.0) if pkm_type2 else 1.0
    return m1 * m2


# Getting type charts
type_chart = load_type_chart(TYPE_CHART_PATH)
df_indexed = type_chart.set_index(["attacking_type", "defending_type"])
type_chart = df_indexed["multiplier"].to_dict()
stats_map = create_stats_map(train_data, test_data)

## 2. Extract Features (both Train and Test Features)

In [ ]:
# --- BASE STATS ---
def get_stats(turn: dict, p2_stats: dict) -> None:
    """
    Store player 2's active pokemon stats in p2_stats if found in stats_map
    """

    pkm2_name = turn["p2_pokemon_state"]["name"]
    if pkm2_name in stats_map:
        stats = stats_map[pkm2_name]["base_stats"]
        p2_stats[pkm2_name] = stats

def feature_base_stats(battle: dict, p2_stats: dict, features: dict) -> None:
    """
    Add base pokemon statistics of:
    - player 1 full team
    - player 2 seen team
    - player 1 mean pokemon stats - player 2 pokemon stats
    """

    p1_team = battle["p1_team_details"]
    features["p1_mean_hp"]  = np.mean([p["base_hp"]  for p in p1_team])
    features["p1_mean_atk"] = np.mean([p["base_atk"] for p in p1_team])
    features["p1_mean_def"] = np.mean([p["base_def"] for p in p1_team])
    features["p1_mean_spa"] = np.mean([p["base_spa"] for p in p1_team])
    features["p1_mean_spd"] = np.mean([p["base_spd"] for p in p1_team])
    features["p1_mean_spe"] = np.mean([p["base_spe"] for p in p1_team])
    features["p1_mean_bst"] = features["p1_mean_hp"] + features["p1_mean_atk"] + features["p1_mean_def"] + features["p1_mean_spa"] + features["p1_mean_spd"] + features["p1_mean_spe"]

    features["p2_mean_hp"]  = np.mean([p["base_hp"]  for p in p2_stats.values()])
    features["p2_mean_atk"] = np.mean([p["base_atk"] for p in p2_stats.values()])
    features["p2_mean_def"] = np.mean([p["base_def"] for p in p2_stats.values()])
    features["p2_mean_spa"] = np.mean([p["base_spa"] for p in p2_stats.values()])
    features["p2_mean_spd"] = np.mean([p["base_spd"] for p in p2_stats.values()])
    features["p2_mean_spe"] = np.mean([p["base_spe"] for p in p2_stats.values()])
    features["p2_mean_bst"] = features["p2_mean_hp"] + features["p2_mean_atk"] + features["p2_mean_def"] + features["p2_mean_spa"] + features["p2_mean_spd"] + features["p2_mean_spe"]

    features["diff_mean_hp"]  = features["p1_mean_hp"]  - features["p2_mean_hp"]
    features["diff_mean_atk"] = features["p1_mean_atk"] - features["p2_mean_atk"]
    features["diff_mean_def"] = features["p1_mean_def"] - features["p2_mean_def"]
    features["diff_mean_spa"] = features["p1_mean_spa"] - features["p2_mean_spa"]
    features["diff_mean_spd"] = features["p1_mean_spd"] - features["p2_mean_spd"]
    features["diff_mean_spe"] = features["p1_mean_spe"] - features["p2_mean_spe"]
    features["diff_mean_bst"] = features["p1_mean_bst"] - features["p2_mean_bst"]


# --- HP LOSS ---
def get_hp_pct_lost(turn: dict, p1_hp_map: dict, p2_hp_map: dict) -> None:
    """
    Track HP loss from one turn for each observed pokemon:
    - remember its latest HP across switches, starting from full HP
    - accumulate HP loss in its three ten-turn intervals
    Healing reduces HP loss.
    """

    name = turn["p1_pokemon_state"]["name"]
    hp = turn["p1_pokemon_state"]["hp_pct"]
    if name not in p1_hp_map:
        p1_hp_map[name] = {"last_hp": 1.0, "losses": [0.0, 0.0, 0.0]}
    pokemon = p1_hp_map[name]
    pokemon["losses"][(turn["turn"] - 1) // 10] += pokemon["last_hp"] - hp
    pokemon["last_hp"] = hp

    name = turn["p2_pokemon_state"]["name"]
    hp = turn["p2_pokemon_state"]["hp_pct"]
    if name not in p2_hp_map:
        p2_hp_map[name] = {"last_hp": 1.0, "losses": [0.0, 0.0, 0.0]}
    pokemon = p2_hp_map[name]
    pokemon["losses"][(turn["turn"] - 1) // 10] += pokemon["last_hp"] - hp
    pokemon["last_hp"] = hp

def feature_hp_lost(p1_hp_map: dict, p2_hp_map: dict, features: dict) -> None:
    """
    Add mean HP loss for both players after all observed turns:
    - sum each pokemon's loss in turns 0-10, 10-20, and 20-30
    - divide each interval total by the final number of observed team pokemon
    Partial intervals (eg. battle ended before 30 turns) are valid too.
    Unobserved intervals remain zero.
    """

    features["p1_hp_pct_lost_0_10"]  = sum(p["losses"][0] for p in p1_hp_map.values()) / len(p1_hp_map)
    features["p1_hp_pct_lost_10_20"] = sum(p["losses"][1] for p in p1_hp_map.values()) / len(p1_hp_map)
    features["p1_hp_pct_lost_20_30"] = sum(p["losses"][2] for p in p1_hp_map.values()) / len(p1_hp_map)

    features["p2_hp_pct_lost_0_10"]  = sum(p["losses"][0] for p in p2_hp_map.values()) / len(p2_hp_map)
    features["p2_hp_pct_lost_10_20"] = sum(p["losses"][1] for p in p2_hp_map.values()) / len(p2_hp_map)
    features["p2_hp_pct_lost_20_30"] = sum(p["losses"][2] for p in p2_hp_map.values()) / len(p2_hp_map)

    features["p1_hp_pct_lost"]   = features["p1_hp_pct_lost_0_10"] + features["p1_hp_pct_lost_10_20"] + features["p1_hp_pct_lost_20_30"]
    features["p2_hp_pct_lost"]   = features["p2_hp_pct_lost_0_10"] + features["p2_hp_pct_lost_10_20"] + features["p2_hp_pct_lost_20_30"]
    features["diff_hp_pct_lost"] = features["p2_hp_pct_lost"] - features["p1_hp_pct_lost"]


# --- TYPE EFFECTIVENESS ---
def get_mults(turn: dict, p1_mults: dict, p2_mults: dict) -> None:
    """
    Update both players' type effectiveness multiplier counts for one turn
    """
    
    mult1, mult2 = 1, 1

    mv1 = turn["p1_move_details"]
    if mv1 != None:
        mv1_category = mv1["category"]
        if mv1_category != "status":
            mv1_type = mv1["type"]
            pkm2_name = turn["p2_pokemon_state"]["name"]
            pkm2_stats = stats_map.get(pkm2_name)
            if pkm2_stats is not None:
                pkm2_types = pkm2_stats.get("types", ["notype", "notype"])
            else:
                pkm2_types = ["notype", "notype"]
            mult1 = get_multiplier(mv1_type, pkm2_types[0], pkm2_types[1])
        p1_mults[mult1] += 1

    mv2 = turn["p2_move_details"]
    if mv2 != None:
        mv2_category = mv2["category"]
        if mv2_category != "status":
            mv2_type = mv2["type"]
            pkm1_name = turn["p1_pokemon_state"]["name"]
            # pkm1 types are always known
            pkm1_types = stats_map[pkm1_name]["types"]
            mult2 = get_multiplier(mv2_type, pkm1_types[0], pkm1_types[1])
        p2_mults[mult2] += 1

def feature_mults(p1_mults: dict, p2_mults: dict, features: dict) -> None:
    """
    Add multiplier counts and weighted scores for both players
    """

    features["p1_0_mult"]     = p1_mults[0]
    features["p1_0.25_mult"]  = p1_mults[0.25]
    features["p1_0.5_mult"]   = p1_mults[0.5]
    features["p1_1_mult"]     = p1_mults[1]
    features["p1_2_mult"]     = p1_mults[2]
    features["p1_4_mult"]     = p1_mults[4]

    features["p1_mult_wmean"] = 0
    mul1_count = 0
    for mult, value in p1_mults.items():
        features["p1_mult_wmean"] += value * MULTS_WEIGHTS[mult]
        mul1_count += 1
    features["p1_mult_wmean"] /= mul1_count

    features["p2_0_mult"]     = p2_mults[0]
    features["p2_0.25_mult"]  = p2_mults[0.25]
    features["p2_0.5_mult"]   = p2_mults[0.5]
    features["p2_1_mult"]     = p2_mults[1]
    features["p2_2_mult"]     = p2_mults[2]
    features["p2_4_mult"]     = p2_mults[4]

    features["p2_mult_wmean"] = 0
    mul2_count = 0
    for mult, value in p2_mults.items():
        features["p2_mult_wmean"] += value * MULTS_WEIGHTS[mult]
        mul2_count += 1
    features["p2_mult_wmean"] /= mul2_count

    features["diff_0_mult"]     = features["p2_0_mult"] - features["p1_0_mult"]
    features["diff_0.25_mult"]  = features["p2_0.25_mult"] - features["p1_0.25_mult"]
    features["diff_0.5_mult"]   = features["p2_0.5_mult"] - features["p1_0.5_mult"]
    features["diff_1_mult"]     = features["p1_1_mult"] - features["p2_1_mult"]
    features["diff_2_mult"]     = features["p1_2_mult"] - features["p2_2_mult"]
    features["diff_4_mult"]     = features["p1_4_mult"] - features["p2_4_mult"]
    features["diff_mult_wmean"] = features["diff_0_mult"] + features["diff_0.25_mult"] + features["diff_0.5_mult"] + features["diff_1_mult"] + features["diff_2_mult"] + features["diff_4_mult"]


# --- STATUSES (ALSO FAINT) ---
def get_statuses(turn: dict, p1_status: dict, p2_status: dict) -> None:
    """
    Accumulate active status turn counts for both players, including fainting
    """
    
    pkm1_state = turn["p1_pokemon_state"]["status"]
    if pkm1_state != "nostatus":
        if pkm1_state not in p1_status:
            p1_status[pkm1_state] = 0
        p1_status[pkm1_state] += 1

    pkm2_state = turn["p2_pokemon_state"]["status"]
    if pkm2_state != "nostatus":
        if pkm2_state not in p2_status:
            p2_status[pkm2_state] = 0
        p2_status[pkm2_state] += 1

def feature_statuses(p1_status_map: dict, p2_status_map: dict, n_turns: int, features: dict) -> None:
    """
    Add status turn totals, weighted sums, and weighted sums per observed turn
    for both players, including fainting. Treat unobserved statuses as zero.
    """

    features["p1_statuses_wsum"] = 0
    for status, count in p1_status_map.items():
        features[f"p1_{status}"] = count
        features["p1_statuses_wsum"] += count * STATUSES_WEIGHTS.get(status, 0)
    features["p1_statuses_wmean"] = features["p1_statuses_wsum"] / n_turns

    features["p2_statuses_wsum"] = 0
    for status, count in p2_status_map.items():
        features[f"p2_{status}"] = count
        features["p2_statuses_wsum"] += count * STATUSES_WEIGHTS.get(status, 0)
    features["p2_statuses_wmean"] = features["p2_statuses_wsum"] / n_turns

    features["diff_statuses_wmean"] = features["p2_statuses_wmean"] - features["p1_statuses_wmean"]


# --- BOOSTS ---
def get_boosts(turn: dict, p1_boosts: dict, p2_boosts: dict) -> None:
    """
    Accumulate signed boost stages per stat for both active pokemon.
    Update both players' totals in place using the current observed turn
    """

    pkm1_boosts = turn["p1_pokemon_state"]["boosts"]
    for boost, value in pkm1_boosts.items():
        p1_boosts[boost] += value

    pkm2_boosts = turn["p2_pokemon_state"]["boosts"]
    for boost, value in pkm2_boosts.items():
        p2_boosts[boost] += value

def feature_boosts(p1_boosts: dict, p2_boosts: dict, n_turns: int, features: dict) -> None:
    """
    Add signed boost stage totals per stat for both players, plus weighted sums
    and weighted sums per observed turn. Negative stages reduce the scores
    """

    features["p1_boosts_wsum"] = 0
    for boost, value in p1_boosts.items():
        features[f"p1_{boost}_boost"] = value
        features["p1_boosts_wsum"] += value * BOOSTS_WEIGHTS[boost]
    features["p1_boosts_wmean"] = features["p1_boosts_wsum"] / n_turns

    features["p2_boosts_wsum"] = 0
    for boost, value in p2_boosts.items():
        features[f"p2_{boost}_boost"] = value
        features["p2_boosts_wsum"] += value * BOOSTS_WEIGHTS[boost]
    features["p2_boosts_wmean"] = features["p2_boosts_wsum"] / n_turns

    features["diff_boosts_wmean"] = features["p1_boosts_wmean"] - features["p2_boosts_wmean"]


# --- EFFECTS FEATURES ---
def get_effects(turn: dict, p1_effects: dict, p2_effects: dict) -> None:
    """
    Accumulate positive and negative effect weights from one turn for both players
    """

    for eff in turn["p1_pokemon_state"]["effects"]:
        w = EFFECT_WEIGHTS.get(eff, 0.0)
        if w > 0:
            p1_effects["pos"] += w
        elif w < 0:
            p1_effects["neg"] += w

    for eff in turn["p2_pokemon_state"]["effects"]:
        w = EFFECT_WEIGHTS.get(eff, 0.0)
        if w > 0:
            p2_effects["pos"] += w
        elif w < 0:
            p2_effects["neg"] += w

def feature_effects(p1_effects: dict, p2_effects: dict, n_turns: int, features: dict) -> None:
    """
    Add weighted effect scores after all observed turns and at the final turn:
    - mean positive, negative, and net effect weights for P1 and P2
    - net weighted effect difference (P1 - P2) across the battle
    """

    features["p1_eff_pos_wmean"] = p1_effects["pos"] / n_turns
    features["p1_eff_neg_wmean"] = p1_effects["neg"] / n_turns
    features["p2_eff_pos_wmean"] = p2_effects["pos"] / n_turns
    features["p2_eff_neg_wmean"] = p2_effects["neg"] / n_turns

    features["p1_eff_net_wmean"]   = features["p1_eff_pos_wmean"] + features["p1_eff_neg_wmean"]
    features["p2_eff_net_wmean"]   = features["p2_eff_pos_wmean"] + features["p2_eff_neg_wmean"]

    features["diff_pos_wmean"]     = features["p1_eff_pos_wmean"] - features["p2_eff_pos_wmean"]
    features["diff_neg_wmean"]     = features["p2_eff_neg_wmean"] - features["p1_eff_neg_wmean"]
    features["diff_eff_net_wmean"] = features["p1_eff_net_wmean"] - features["p2_eff_net_wmean"]


# --- TEAM SNAPSHOT AT LAST TURN ---
def get_last_seen(turn: dict, p1_last_seen: dict, p2_last_seen: dict) -> None:
    """
    Store each observed pokemon's latest HP and status stages.
    Update both players' maps in place; retain switched-out pokemon's snapshots.
    """

    # Effects (many non really all) and boosts goes away when a pokemon leaves the arena
    # So they are not tracked (simplification on effects)

    pkm1 = turn["p1_pokemon_state"]
    p1_last_seen[pkm1["name"]] = {
        "hp_pct": pkm1["hp_pct"],
        "status": pkm1["status"]
    }

    pkm2 = turn["p2_pokemon_state"]
    p2_last_seen[pkm2["name"]] = {
        "hp_pct": pkm2["hp_pct"],
        "status": pkm2["status"]
    }

def feature_last_seen(last_turn: dict, p1_last_seen: dict, p2_last_seen: dict, features: dict) -> None:
    """
    Add latest-known team HP, status, boost, and effect means and P1 - P2 differences.
    Divide by six; assume unseen pokemon have full HP and no status, effects, or boosts.
    Benched pokemon retain their last observed data, which may be older.
    """

    features["p1_lst_hp_mean"] = 1.0 * (TEAM_SIZE - len(p1_last_seen))
    features["p1_lst_statuses_wmean"] = 0.0
    features["p1_lst_effects_wmean"] = 0.0
    features["p1_lst_boosts_wmean"] = 0.0
    for pkm1 in p1_last_seen.values():
        features["p1_lst_hp_mean"] += pkm1["hp_pct"]
        features["p1_lst_statuses_wmean"] += STATUSES_WEIGHTS.get(pkm1["status"], 0)
    lst_pkm1_effects = last_turn["p1_pokemon_state"]["effects"]
    for eff in lst_pkm1_effects:
        features["p1_lst_effects_wmean"] += EFFECT_WEIGHTS.get(eff, 0)
    lst_pkm1_boosts = last_turn["p1_pokemon_state"]["boosts"]
    for bst, value in lst_pkm1_boosts.items():
        features["p1_lst_boosts_wmean"] += value * BOOSTS_WEIGHTS.get(bst, 0)
    features["p1_lst_hp_mean"]        /= TEAM_SIZE
    features["p1_lst_statuses_wmean"] /= TEAM_SIZE
    features["p1_lst_effects_wmean"]  /= TEAM_SIZE
    features["p1_lst_boosts_wmean"]   /= TEAM_SIZE

    features["p2_lst_hp_mean"] = 1.0 * (TEAM_SIZE - len(p2_last_seen))
    features["p2_lst_statuses_wmean"] = 0.0
    features["p2_lst_effects_wmean"] = 0.0
    features["p2_lst_boosts_wmean"] = 0.0
    for pkm2 in p2_last_seen.values():
        features["p2_lst_hp_mean"] += pkm2["hp_pct"]
        features["p2_lst_statuses_wmean"] += STATUSES_WEIGHTS.get(pkm2["status"], 0)
    lst_pkm2_effects = last_turn["p2_pokemon_state"]["effects"]
    for eff in lst_pkm2_effects:
        features["p2_lst_effects_wmean"] += EFFECT_WEIGHTS.get(eff, 0)
    lst_pkm2_boosts = last_turn["p2_pokemon_state"]["boosts"]
    for bst, value in lst_pkm2_boosts.items():
        features["p2_lst_boosts_wmean"] += value * BOOSTS_WEIGHTS.get(bst, 0)
    features["p2_lst_hp_mean"]        /= TEAM_SIZE
    features["p2_lst_statuses_wmean"] /= TEAM_SIZE
    features["p2_lst_effects_wmean"]  /= TEAM_SIZE
    features["p2_lst_boosts_wmean"]   /= TEAM_SIZE

    features["diff_lst_hp_mean"]        = features["p1_lst_hp_mean"] - features["p2_lst_hp_mean"]
    features["diff_lst_statuses_wmean"] = features["p2_lst_statuses_wmean"] - features["p1_lst_statuses_wmean"]
    features["diff_lst_effects_wmean"]  = features["p1_lst_effects_wmean"] - features["p2_lst_effects_wmean"]
    features["diff_lst_boosts_wmean"]   = features["p1_lst_boosts_wmean"] - features["p2_lst_boosts_wmean"]

In [ ]:
# --- EXTRACT FEATURES ---
def extract_features(data: list) -> pd.DataFrame:
    """
    Extract the features:
    1. base stats
    2. HP loss in ten-turn intervals
    3. type effectiveness multiplier counts
    4. statuses
    5. boosts
    6. effects
    7. last seen
    """

    print("Extracting features...")
    feature_list = []

    for battle in data:
        features = {"battle_id" : battle["battle_id"]}

        # Keep the identifier and target for the training and submission cells
        if "player_won" in battle:
            features["player_won"] = battle["player_won"]


        # Initializations
        p1_hp_map, p2_hp_map = {}, {}
        p1_mults = {mult: 0 for mult in MULTS_WEIGHTS}
        p2_mults = {mult: 0 for mult in MULTS_WEIGHTS}
        p1_status, p2_status = {}, {}
        p1_boosts = {boost: 0 for boost in BOOSTS_WEIGHTS}
        p2_boosts = {boost: 0 for boost in BOOSTS_WEIGHTS}
        p1_effects = {"pos": 0.0, "neg": 0.0}
        p2_effects = {"pos": 0.0, "neg": 0.0}
        p1_last_seen, p2_last_seen = {}, {}
        p2_stats = {}

        timeline = battle["battle_timeline"]
        for turn in battle["battle_timeline"]:
            get_stats(turn, p2_stats)
            get_hp_pct_lost(turn, p1_hp_map, p2_hp_map)
            get_mults(turn, p1_mults, p2_mults)
            get_statuses(turn, p1_status, p2_status)
            get_boosts(turn, p1_boosts, p2_boosts)
            get_effects(turn, p1_effects, p2_effects)
            get_last_seen(turn, p1_last_seen, p2_last_seen)

        last_turn = timeline[-1]
        # 1. Base stats
        feature_base_stats(battle, p2_stats, features)
        # 2. HP loss in ten-turn intervals
        feature_hp_lost(p1_hp_map, p2_hp_map, features)
        # 3. Type effectiveness multiplier counts
        feature_mults(p1_mults, p2_mults, features)
        # 4. Statuses
        feature_statuses(p1_status, p2_status, len(timeline), features)
        # 5. Boosts
        feature_boosts(p1_boosts, p2_boosts, len(timeline), features)
        # 6. Effects
        feature_effects(p1_effects, p2_effects, len(timeline), features)
        # 7. Last seen
        feature_last_seen(last_turn, p1_last_seen, p2_last_seen, features)


        feature_list.append(features)

    print(f"SUCCESS: extracted {len(feature_list[0])} features.")
    return pd.DataFrame(feature_list).fillna(0)


# Extracting training and test features
train_df = extract_features(train_data)
print(f"Training features: {train_df.shape}.")
display(train_df.head())

test_df = extract_features(test_data)
print(f"Test features: {test_df.shape}.")
# display(test_df.head())

## 3. Training

### Models

In [ ]:
# --- BUILD MODEL ---
def make_lr(random_state: int): # -> Pipeline:
    """
    Build an instance of logistic_regression with seed `random_state`
    """
    
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, C=1.0, random_state=random_state)
    )

def make_rf(random_state: int) -> RandomForestClassifier:
    """
    Build an instance of `random_forest` with seed `random_state`
    """

    return RandomForestClassifier(
        n_estimators=150,
        max_depth=8,
        min_samples_split=5,
        random_state=random_state,
        n_jobs=-1
    )

def make_lgbm(random_state: int) -> LGBMClassifier:
    """
    Build an instance of `lightgbm` with seed `random_state`
    """
        
    return LGBMClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.04,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        verbose=-1,
        random_state=random_state
    )

def build_model(model_type: str = MODEL_TYPE, random_state: int = RANDOM_STATE): # -> BaseEstimator:
    """
    Build an instance of `model_type` with seed `random_state`
    """

    match model_type:
        case "logistic_regression":
            return make_lr(random_state)

        case "random_forest":
            return make_rf(random_state)

        case "lightgbm":
            return make_lgbm(random_state)

        case "ensemble":
            return VotingClassifier(
                estimators=[
                    ("lgb", make_lgbm(random_state)),
                    ("rf", make_rf(random_state)),
                    ("lr", make_lr(random_state)),
                ],
                voting="soft",
                weights=[0.5, 0.3, 0.2],
            )

        case _:
            raise ValueError(f"ERROR: Unavailable model: {model_type}.")

### Perfomance Metrics

In [ ]:
# --- METRICS ---
def compute_metrics(y_true: pd.Series | np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray) -> dict:
    """
    Return accuracy, precision, recall, F1, and ROC-AUC for binary predictions.
    y_true and y_pred contain 0/1 labels.
    y_prob contains probabilities of class 1
    """

    return {
        # (TP + TN) / total
        "accuracy": accuracy_score(y_true, y_pred),
        # TP / (TP + FP)
        "precision": precision_score(y_true, y_pred, zero_division=0),
        # TP / (TP + FN)
        "recall": recall_score(y_true, y_pred, zero_division=0),
        # 2 * (precision * recall) / (precision + recall) = 2 * TP / (2 * TP + FP + FN)
        "f1": f1_score(y_true, y_pred, zero_division=0),
        # Area Under the Receiver Operating Characteristic Curve
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

### Training

In [ ]:
# --- TRAINING ---
def train_and_evaluate_cv(X: pd.DataFrame, y: pd.Series, X_test: pd.DataFrame, model_type: str = MODEL_TYPE, k: int = K_FOLDS, n_repeats: int = N_REPEATS, random_state: int = RANDOM_STATE) -> tuple:
    """
    Repeated Stratified K-Fold Cross-Validation:
    1. Computes metrics on each of the k * n_repeats validation folds.
    2. Computes Out-Of-Fold (OOF) probabilities, averaged across repeats.
    3. Averages the test-set probabilities over all k * n_repeats models.
    """
    
    # Shuffle of K-Fold across n_repeats from a fixed random seed (random_state=42, n_repeats=3 -> random_states = [42, 43, 44]) 
    rskf = RepeatedStratifiedKFold(n_splits=k, n_repeats=n_repeats, random_state=random_state)
    n_total = k * n_repeats

    # Initialize arrays to store Out-Of-Fold (OOF) probabilities and test set probabilities
    oof_probs  = np.zeros(len(X))        # Initialize the OOF probabilities for each battle in the training set (len(X) = number of battles in the training set)
    test_probs = np.zeros(len(X_test))  # Initialize the test set probabilities for each battle in the test set (as above)
    print(f"Starting {n_repeats}x Stratified {k}-Fold CV with model: {model_type}.")

    fold_metrics_list = []
    for i, (train_idx, val_idx) in enumerate(rskf.split(X, y)):
        repeat, fold = divmod(i, k) # Get the current repeat (i) and fold number (k)
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx] # iloc -> select the rows using numerical positions (X train rows and X validation rows)
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx] # target variable for training and validation

        model = build_model(model_type, random_state=random_state)
        model.fit(X_train, y_train) # Train the model on the training fold

        # Prediction on validation fold
        val_prob = model.predict_proba(X_val)[:, 1] # Getting the probabilities for the "player_won" (1) (0 = "player_lost") for the validation set (float values)
        val_pred = (val_prob >= 0.5).astype(int)    # Tresholding the probabilities to get binary predictions (1 if prob >= 0.5, else 0)
        oof_probs[val_idx] += val_prob / n_repeats  # Accumulate the OOF probabilities for the validation fold, averaged across repeats

        metrics = compute_metrics(y_val, val_pred, val_prob)
        fold_metrics_list.append(metrics)

        # print(f"Repeat {repeat+1}/{n_repeats} | Fold {fold+1}/{k} -> " f"Accuracy: {metrics['accuracy']:.4f} | F1: {metrics['f1']:.4f} | " f"ROC-AUC: {metrics['roc_auc']:.4f}")
        metrics_text = "".join(f"{name}: {value:.4f} | " for name, value in metrics.items())
        print(f"Repeat {repeat+1}/{n_repeats} | Fold {fold+1}/{k} -> {metrics_text}")

        # Aggregate test set probabilities across all folds and repeats
        test_probs += model.predict_proba(X_test)[:, 1] / n_total

    # Merged statistics on K fold
    mean_acc = np.mean([m["accuracy"] for m in fold_metrics_list])
    std_acc  = np.std( [m["accuracy"] for m in fold_metrics_list])
    mean_f1  = np.mean([m["f1"]       for m in fold_metrics_list])
    mean_auc = np.mean([m["roc_auc"]  for m in fold_metrics_list])

    # Overall performance Out-Of-Fold
    oof_pred        = (oof_probs >= 0.5).astype(int)
    overall_oof_acc = accuracy_score(y, oof_pred)
    overall_oof_auc = roc_auc_score(y, oof_probs)

    print(f"Aggregated cv results ({n_repeats}x{k} folds = {n_total} evaluations):")
    print(f"Mean accuracy    : {mean_acc:.4f} (+/- {std_acc:.4f})")
    print(f"Harmonic mean F1 : {mean_f1:.4f}")
    print(f"Mean ROC-AUC     : {mean_auc:.4f}")
    print(f"OOF Accuracy     : {overall_oof_acc:.4f}")
    print(f"OOF ROC-AUC      : {overall_oof_auc:.4f}")

    test_predictions = (test_probs >= 0.5).astype(int)
    return test_predictions, test_probs


# Features excluding the ID and target columns
feature_cols = [
    c for c in train_df.columns
    if c not in ["battle_id", "player_won"] and c not in DISABLED_FEATURES
]

print(f"featurs: {len(feature_cols)}.")
X_train = train_df[feature_cols]
y_train = train_df["player_won"]
X_test  = test_df[feature_cols]

# Training, K-Fold Validation, and Ensemble Prediction
test_predictions, test_probabilities = train_and_evaluate_cv(
    X=X_train,
    y=y_train,
    X_test=X_test,
    model_type=MODEL_TYPE,
    k=K_FOLDS,
    random_state=RANDOM_STATE,
)

## 4. Submission File

In [ ]:
def predict_and_save(predictions: np.ndarray, test_battle_ids: pd.Series, output_path: str = SUBMISSION_PATH, sample_path: str = SAMPLE_SUBMISSION_PATH) -> None:
    """
    Format and save the submission file using the required template
    """
    
    print(f"Saving predictions to '{output_path}'...")
    sample_submission = pd.read_csv(sample_path)
    expected_columns = ["battle_id", "player_won"]
    if list(sample_submission.columns) != expected_columns:
        raise ValueError(f"ERROR: expected submission columns {expected_columns}, got {list(sample_submission.columns)}")

    predictions_by_id = pd.Series(predictions, index=test_battle_ids)
    sample_submission["player_won"] = sample_submission["battle_id"].map(predictions_by_id)

    if sample_submission["player_won"].isna().any():
        raise ValueError(f"ERROR: some battle IDs in the '{sample_path}' have no matching prediction.")

    sample_submission["player_won"] = sample_submission["player_won"].astype(int)
    sample_submission.to_csv(output_path, index=False)

    print(f"SUCCESS: file '{output_path}' created.")
    print(sample_submission.head(10))
    print(f"Prediction distribution (0 vs 1): {dict(sample_submission['player_won'].value_counts())}")


# Submission file
predict_and_save(test_predictions, test_df["battle_id"], SUBMISSION_PATH, SAMPLE_SUBMISSION_PATH)

## Debug

In [ ]:
# --- BATTLE EXAMPLE ---
def print_battle(data, battle_idx=0):
    """
    Prints the whole battle at index `battle_idx`
    """
    print(f"\n-----------------print_battle()-----------------")

    battle = data[battle_idx]
    output_path = f"../temp/battle_{battle_idx}.json"
    with open(output_path, "w") as f:
        json.dump(battle, f, indent=4)

    print(f"SUCCESS: battle saved to {output_path}.")


def diagnose_type_chart(type_chart, stats_map, train_data):
    """Check that type names in the chart match the training data."""
    print(f"\n-----------------diagnose_type_chart()-----------------")

    chart_attack = {a for a, d in type_chart.keys()}
    chart_defend = {d for a, d in type_chart.keys()}

    pokemon_types = {pokemon_type for pokemon in stats_map.values() for pokemon_type in pokemon["types"] if pokemon_type}
    move_types, move_categories = set(), set()
    for battle in train_data:
        for turn in battle["battle_timeline"]:
            for key in ("p1_move_details", "p2_move_details"):
                move = turn[key]
                if move is not None:
                    move_types.add(move["type"])
                    move_categories.add(move["category"])

    print("Tipi in tabella (attacco):", sorted(map(repr, chart_attack)))
    print("Tipi in tabella (difesa) :", sorted(map(repr, chart_defend)))
    print("Tipi Pokemon nei dati    :", sorted(map(repr, pokemon_types)))
    print("Tipi mosse nei dati      :", sorted(map(repr, move_types)))
    print("Categorie mosse          :", sorted(map(repr, move_categories)))
    print("\nTipi di mosse NON presenti in tabella  :", move_types - chart_attack)
    print("Tipi di Pokemon NON presenti in tabella:", pokemon_types - chart_defend)
    print("Tipi di Pokemon NON presenti in tabella:", pokemon_types - chart_defend)

    print("\nProva su coppie (tipo mossa, tipo Pokemon):")
    for move_type in sorted(move_types)[:3]:
        for pokemon_type in sorted(pokemon_types)[:3]:
            value = type_chart.get((move_type, pokemon_type), "1.00")
            print(f"  {move_type!r:12} vs {pokemon_type!r:12} -> {value}")


def diagnose_fnt_features(train_df):
    """Summarize the aggregate FNT features."""
    print(f"\n-----------------diagnose_fnt_features()-----------------")
    print(train_df[["p1_fnt", "p2_fnt"]].describe())
    print(train_df.groupby("player_won")[["p1_fnt", "p2_fnt"]].mean())


def diagnose_ko_records(train_data):
    """Inspect how fainted Pokemon are recorded in battle timelines."""
    print(f"\n-----------------diagnose_ko_records()-----------------")

    rows = []
    for battle in train_data:
        timeline = battle["battle_timeline"]
        row = {}
        for player in (1, 2):
            states = [turn[f"p{player}_pokemon_state"] for turn in timeline]
            row[f"p{player}_turns_status_fnt"] = sum(state["status"] == "fnt" for state in states)
            row[f"p{player}_turns_hp0"] = sum(state["hp_pct"] == 0 for state in states)
            row[f"p{player}_distinct_names"] = len({state["name"] for state in states})
            row[f"p{player}_distinct_ko"] = len({state["name"] for state in states if state["hp_pct"] == 0 or state["status"] == "fnt"})
        rows.append(row)
    print(pd.DataFrame(rows).mean().round(3).to_string())

    battle = next(battle for battle in train_data if any(turn["p2_pokemon_state"]["hp_pct"] == 0 for turn in battle["battle_timeline"]))
    for index, turn in enumerate(battle["battle_timeline"]):
        state1, state2 = turn["p1_pokemon_state"], turn["p2_pokemon_state"]
        print(f"{index:2d} | {state1['name']:12} {state1['hp_pct']:.2f} {state1['status']:9} | " f"{state2['name']:12} {state2['hp_pct']:.2f} {state2['status']}")


def diagnose_species_stats(train_data, test_data):
    """Check consistency of base stats and levels across the datasets."""
    from collections import Counter, defaultdict

    print(f"\n-----------------diagnose_species_stats()-----------------")

    base_stats = ["base_hp", "base_atk", "base_def", "base_spa", "base_spd", "base_spe"]
    stats_seen = defaultdict(set)
    levels_seen = defaultdict(Counter)

    for data in (train_data, test_data):
        for battle in data:
            for pokemon in battle["p1_team_details"] + [battle["p2_lead_details"]]:
                stats_seen[pokemon["name"]].add(tuple(pokemon[key] for key in base_stats))
                levels_seen[pokemon["name"]][pokemon["level"]] += 1

    print("Specie viste:", len(stats_seen))
    inconsistent = {name for name, stats in stats_seen.items() if len(stats) > 1}
    print("Specie con piu' set di statistiche base:", len(inconsistent), list(inconsistent)[:5])

    total_levels = Counter()
    for counts in levels_seen.values():
        total_levels.update(counts)
    print("Distribuzione livelli:", sorted(total_levels.items()))

    not_level_100 = {name: dict(counts) for name, counts in levels_seen.items() if set(counts) != {100}}
    print("Specie con almeno un livello != 100:", len(not_level_100))
    for name, counts in list(not_level_100.items())[:15]:
        print(" ", name, counts)

    seen_p2 = {turn["p2_pokemon_state"]["name"] for battle in train_data + test_data for turn in battle["battle_timeline"]}
    print("Specie di P2 senza statistiche:", seen_p2 - set(stats_seen))


def diagnose_distinct_fnt(train_data):
    """Count distinct fainted Pokemon for P1 and P2."""
    print(f"\n-----------------diagnose_distinct_fnt()-----------------")

    p1_fnt_names_list, p2_fnt_names_list = [], []
    for battle in train_data:
        p1_fnt_names = set()
        p2_fnt_names = set()

        for turn in battle["battle_timeline"]:
            state1 = turn["p1_pokemon_state"]
            state2 = turn["p2_pokemon_state"]

            if state1["status"] == "fnt" or state1["hp_pct"] == 0:
                p1_fnt_names.add(state1["name"])
            if state2["status"] == "fnt" or state2["hp_pct"] == 0:
                p2_fnt_names.add(state2["name"])

        p1_fnt_names_list.append(len(p1_fnt_names))
        p2_fnt_names_list.append(len(p2_fnt_names))

    print("Media Pokemon distinti con status=='fnt' o hp==0 -> " f"P1: {np.mean(p1_fnt_names_list):.4f} | " f"P2: {np.mean(p2_fnt_names_list):.4f}")


def print_features_correlation(X_train, y_train):
    print(f"\n-----------------print_features_correlation()-----------------")
    print(f"\nCORRELAZIONE TRA LE FEATURE E IL TARGET (player_won):")
    corr = X_train.corrwith(y_train).sort_values()
    print(corr.head(80))  # stampa tutte e 80 le features

    corr = X_train.corrwith(y_train)
    nan_cols = corr[corr.isna()].index.tolist()
    print(f"Feature con correlazione NaN: {len(nan_cols)}")
    print(X_train[nan_cols].nunique())  # 1 = colonna costante
    print(X_train[nan_cols].iloc[0])  # il valore che ripetono

    corr = corr.dropna().sort_values()
    print(corr.to_string())  # tutte le altre, nessun taglio


if DEBUG:
    print_battle(train_data)
    diagnose_type_chart(type_chart, stats_map, train_data)
    diagnose_fnt_features(train_df)
    diagnose_ko_records(train_data)
    diagnose_species_stats(train_data, test_data)
    diagnose_distinct_fnt(train_data)
    print_features_correlation(X_train, y_train)